# Task 3

In [3]:
%pip install -r requirements.txt

   ---------------------------------------- 0.0/11.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.0 MB ? eta -:--:--
    --------------------------------------- 0.3/11.0 MB ? eta -:--:--
   - -------------------------------------- 0.5/11.0 MB 1.2 MB/s eta 0:00:09
   - -------------------------------------- 0.5/11.0 MB 1.2 MB/s eta 0:00:09
   - -------------------------------------- 0.5/11.0 MB 1.2 MB/s eta 0:00:09
   - -------------------------------------- 0.5/11.0 MB 1.2 MB/s eta 0:00:09
   - -------------------------------------- 0.5/11.0 MB 1.2 MB/s eta 0:00:09
   - -------------------------------------- 0.5/11.0 MB 1.2 MB/s eta 0:00:09
   - -------------------------------------- 0.5/11.0 MB 1.2 MB/s eta 0:00:09
   - -------------------------------------- 0.5/11.0 MB 1.2 MB/s eta 0:00:09
   - -------------------------------------- 0.5/11.0 MB 1.2 MB/s eta 0:00:09
   - ----------------------

  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
hazm 0.10.0 requires numpy==1.24.3, but you have numpy 2.2.6 which is incompatible.


In [5]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity, linear_kernel

## Dataset

In [6]:
DATA_PATH = "tmdb_5000_movies.csv"   # adjust path if the file is elsewhere

df_raw = pd.read_csv(DATA_PATH)
print("Raw dataset shape:", df_raw.shape)

df = df_raw[["title", "overview"]].copy()

# Drop rows where 'overview' is missing or NaN (as instructed)
before = len(df)
df.dropna(subset=["overview"], inplace=True)
df.reset_index(drop=True, inplace=True)
after = len(df)

print(f"Dropped {before - after} rows with missing overview.")
print("Final dataset shape:", df.shape)
df.head(5)

Raw dataset shape: (4803, 20)
Dropped 3 rows with missing overview.
Final dataset shape: (4800, 2)


,title,overview
0,Avatar,"In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora o..."
1,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, has come back to life and is hea..."
2,Spectre,A cryptic message from Bond’s past sends him on a trail to uncover a siniste...
3,The Dark Knight Rises,"Following the death of District Attorney Harvey Dent, Batman assumes respons..."
4,John Carter,"John Carter is a war-weary, former military captain who's inexplicably trans..."


## Vectorization

### Handle duplicate titles

In [7]:
duplicate_titles = df["title"][df["title"].duplicated()].unique()
print(f"Number of duplicate movie titles: {len(duplicate_titles)}")
if len(duplicate_titles) > 0:
    print("Examples:", duplicate_titles[:5])

Number of duplicate movie titles: 3
Examples: ['The Host' 'Out of the Blue' 'Batman']


In [9]:
tfidf = TfidfVectorizer(stop_words="english")

tfidf_matrix = tfidf.fit_transform(df["overview"])

print("TF-IDF matrix shape:", tfidf_matrix.shape)
print(f"{tfidf_matrix.shape[0]} movies, {tfidf_matrix.shape[1]} unique vocabulary terms")


TF-IDF matrix shape: (4800, 20978)
4800 movies, 20978 unique vocabulary terms


### Calculating the cosine similarity matrix

In [11]:
cosine_sim_matrix = linear_kernel(tfidf_matrix, tfidf_matrix)

print("Cosine similarity matrix shape:", cosine_sim_matrix.shape)

# Build a reverse lookup
# For duplicate titles, keep the index of the FIRST occurrence.
title_to_index = pd.Series(df.index, index=df["title"]).drop_duplicates(keep="first")

Cosine similarity matrix shape: (4800, 4800)


## Recommended browser & Cosine Similarity

In [12]:
def get_movie_recommendation(movie_title: str, top_n: int = 5) -> pd.DataFrame:
    """
    Given a movie title, return the top `top_n` most similar movies
    based on cosine similarity between TF-IDF vectors of their overviews.

    Parameters
    ----------
    movie_title : str
        The exact title of the movie to base recommendations on.
    top_n : int, default 5
        Number of similar movies to return.

    Returns
    -------
    pd.DataFrame
        Columns: 'title', 'similarity_score' - sorted by similarity, descending.
        Returns an empty DataFrame with a printed message if the title is not found.
    """
    if movie_title not in title_to_index:
        print(f"Movie '{movie_title}' not found in the dataset. "
              f"Check spelling/capitalization (titles are case-sensitive).")
        return pd.DataFrame(columns=["title", "similarity_score"])

    idx = title_to_index[movie_title]

    # Get similarity scores of this movie with every other movie
    similarity_scores = list(enumerate(cosine_sim_matrix[idx]))

    # Sort movies by similarity score, in descending order
    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)

    # Exclude the movie itself (always the top match with score 1.0)
    similarity_scores = [s for s in similarity_scores if s[0] != idx]

    top_results = similarity_scores[:top_n]

    recommended_indices = [i for i, score in top_results]
    recommended_scores = [score for i, score in top_results]

    result_df = pd.DataFrame({
        "title": df["title"].iloc[recommended_indices].values,
        "similarity_score": recommended_scores,
    })

    return result_df


print("Recommendation function is ready.")


Recommendation function is ready.


## Output

In [15]:
test_title = "The Dark Knight"
recommendations = get_movie_recommendation(test_title, top_n=5)

print(f"Top 5 movies similar to '{test_title}':\n")
print(recommendations.to_string(index=False))


Top 5 movies similar to 'The Dark Knight':

                                  title  similarity_score
                  The Dark Knight Rises          0.301516
                         Batman Returns          0.249434
Batman: The Dark Knight Returns, Part 2          0.224521
                         Batman Forever          0.214072
                                 Batman          0.182334


### Result: Yes. All 5 suggested films are both from the Batman series and fall into the action/crime genre.

### Try a few more test cases to explore the engine

In [16]:
for movie in ["Avatar", "Inception", "Spectre"]:
    print(f"\n=== Recommendations for '{movie}' ===")
    print(get_movie_recommendation(movie, top_n=5).to_string(index=False))


=== Recommendations for 'Avatar' ===
               title  similarity_score
           Apollo 18          0.186806
        The American          0.170153
          The Matrix          0.132308
The Inhabited Island          0.123932
    Tears of the Sun          0.122487

=== Recommendations for 'Inception' ===
                             title  similarity_score
                            Cypher          0.095411
Mission: Impossible - Rogue Nation          0.094566
                        Stone Cold          0.086964
              Central Intelligence          0.082533
                   Pitch Perfect 2          0.078908

=== Recommendations for 'Spectre' ===
                title  similarity_score
Never Say Never Again          0.256957
From Russia with Love          0.198854
          Thunderball          0.189262
           Safe Haven          0.168193
    Quantum of Solace          0.165085
